# Clustering — Exercises

**Companion to deck 07 (Clustering & Dimensionality).** Find groups in unlabeled data with KMeans.

Real NOAI task example: Kazakhstan TST 2025 'Player Clustering' — group players by stats without knowing roles in advance.

<a href="https://colab.research.google.com/github/Petkub/MachineLearningLab/blob/main/colab_exercises/07_clustering.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

iris = load_iris(as_frame=True)
df = iris.frame
# We pretend we don't know the species — that's what 'unsupervised' means.
X = df.drop('target', axis=1)
y_true = df['target']   # only for verification at the end
print(X.shape)
X.head()

---
## Problem 01 — fit your first KMeans

Tasks:
1. Build a `KMeans(n_clusters=3, random_state=42, n_init=10)`.
2. Fit on `X` and predict cluster labels into `labels`.
3. Print how many points landed in each cluster (use `pd.Series(labels).value_counts()`).

In [ ]:
# TODO
km = ...
labels = ...

print(pd.Series(labels).value_counts())

In [ ]:
assert len(set(labels)) == 3, 'expected 3 cluster ids'
assert len(labels) == len(X)
print('Q1 ok')

<details><summary>Hint</summary>

`KMeans(n_clusters=k, random_state=42, n_init=10)` — `n_init` retries from different seeds and keeps the best (silences a deprecation warning too). For "fit + return labels in one shot," sklearn estimators expose a `.fit_predict(X)` method that does both.

</details>

---
## Problem 02 — scaling matters here too

KMeans uses Euclidean distance. If one feature has a huge range, it dominates clustering.

Tasks:
1. Scale `X` with `StandardScaler` → `X_scaled`.
2. Fit KMeans on `X_scaled`. Save labels in `labels_scaled`.
3. Are clusters the same? Compare counts — should still be roughly 50/50/50 for iris.

In [ ]:
# TODO
scaler = ...
X_scaled = ...
labels_scaled = ...

print('un-scaled:', sorted(pd.Series(labels).value_counts().tolist(), reverse=True))
print('scaled:   ', sorted(pd.Series(labels_scaled).value_counts().tolist(), reverse=True))

In [ ]:
assert X_scaled.shape == X.shape
assert abs(X_scaled.mean()) < 0.01, 'scaled data should have mean ~0'
assert len(set(labels_scaled)) == 3
print('Q2 ok')

---
## Problem 03 — choose k with the elbow method

If you don't know the right `n_clusters`, try several and plot the **inertia** (sum of squared distances to centroid). Look for the "elbow" — where the curve bends.

Tasks:
1. For k from 1 to 8, fit KMeans on `X_scaled`, record `km.inertia_`.
2. Plot k vs inertia.
3. Eyeball the elbow.

In [ ]:
# TODO
ks = list(range(1, 9))
inertias = []
# fit KMeans for each k, append inertia

plt.figure(figsize=(7, 4))
plt.plot(ks, inertias, 'o-')
plt.xlabel('k'); plt.ylabel('inertia')
plt.title('Elbow plot — pick the bend')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
assert len(inertias) == 8
# inertia must monotonically decrease as k grows
for i in range(7):
    assert inertias[i] >= inertias[i+1] - 1e-6, f'inertia not decreasing at k={ks[i+1]}'
print('Q3 ok — eyeball: where does the curve bend?')

---
## Problem 04 — silhouette score (numeric way to compare k)

Higher silhouette = clusters are tight and well-separated. Pick the k with the highest score.

Tasks:
1. For k from 2 to 8, compute silhouette score on `X_scaled`.
2. Plot k vs silhouette.
3. Save the best k in `best_k`.

In [ ]:
# TODO
ks = list(range(2, 9))
sils = []
# for each k: fit KMeans, get labels, compute silhouette_score(X_scaled, labels)

best_k = ...

plt.figure(figsize=(7, 4))
plt.plot(ks, sils, 'o-')
plt.xlabel('k'); plt.ylabel('silhouette score')
plt.title(f'Best k = {best_k}')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
assert best_k in ks
assert sils[ks.index(best_k)] == max(sils)
print(f'Q4 ok — best_k = {best_k}, silhouette = {max(sils):.3f}')

<details><summary>Hint</summary>

Inside the loop: build a fresh KMeans for each `k`, fit it on `X_scaled`, then call the silhouette function (it takes the data + the labels — `km.labels_` after fitting). Append the score. Outside the loop: `numpy` has an `argmax` for "where is the maximum" — use it on `sils` to find the index, then look up the matching `k` in `ks`.

</details>

---
## Problem 05 — visualize clusters in 2D with PCA

Iris has 4 features — can't plot 4D. Use PCA to project to 2D, color by cluster label.

Tasks:
1. Fit `PCA(n_components=2)` on `X_scaled` → `X_pca`.
2. Fit KMeans with `best_k` clusters on `X_scaled`. Get `labels`.
3. Scatter plot `X_pca[:, 0]` vs `X_pca[:, 1]`, color by `labels`.

In [ ]:
# TODO
pca = ...
X_pca = ...
km = ...
labels = ...

plt.figure(figsize=(8, 6))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap='viridis', s=50, alpha=0.7)
plt.xlabel('PC 1'); plt.ylabel('PC 2')
plt.title(f'KMeans (k={best_k}) — projected onto first 2 PCs')
plt.colorbar(label='cluster')
plt.show()

<details><summary>Hint</summary>

Most sklearn transformers expose a `.fit_transform(X)` shortcut — fits the model, then returns the transformed array, in one call. `PCA(n_components=2).fit_transform(X_scaled)` gives you a `(150, 2)` array. Same one-shot helper exists on `StandardScaler`, etc.

</details>

---
## Problem 06 — sanity check against the (hidden) truth

We had `y_true` set aside. Let's see how well our unsupervised clustering recovered the species.

Note: cluster IDs are arbitrary (cluster 0 isn't necessarily species 0). To check, build a confusion-style cross-tab.

Tasks:
1. Build `pd.crosstab(y_true, labels)`. 
2. Each row should be dominated by ONE cluster id (most cells small, one big).

In [ ]:
# TODO
ct = ...
print(ct)

# extra: what fraction of points landed in the most common cluster for their species?
purity = (ct.max(axis=1).sum()) / len(y_true)
print(f'\npurity ≈ {purity:.3f}')

In [ ]:
assert ct.shape == (3, best_k)
assert purity > 0.7, 'KMeans should recover iris species reasonably well'
print('Q6 ok')

---
## Problem 07 — find the bugs in this clustering

The cell below tries to cluster iris into 3 groups and report purity. It runs but the purity is suspiciously low (~0.40, near random). Find and fix **three bugs**.

Things to look for: forgot to scale, picked k from the wrong metric, used the labels of the wrong fit when scoring.

In [ ]:
# BROKEN — fix three things, then save the corrected purity in `purity_q7`.
#
# Bugs to find:
#  - clustered on raw features (one feature has a much larger range — distance is dominated)
#  - n_clusters chosen by lowest inertia (always picks the largest k, useless)
#  - the crosstab uses labels from a different fit than the one we just made

# Bug 1: should fit on a scaled copy
km_bad = KMeans(n_clusters=2, random_state=42, n_init=10)   # Bug 2: wrong k
labels_bad = km_bad.fit_predict(X)

# pretend earlier code computed labels_other from a different clustering
labels_other = KMeans(n_clusters=4, random_state=0, n_init=10).fit_predict(X)

# Bug 3: this crosstab uses labels_other, not the k=2 fit above
ct_bad = pd.crosstab(y_true, labels_other)
purity_q7 = ct_bad.max(axis=1).sum() / len(y_true)

print('purity:', round(purity_q7, 3))

In [ ]:
assert purity_q7 > 0.85, f'still buggy — got {purity_q7}, expected ~0.83+ after all 3 fixes'
print('Q7 ok — purity:', round(purity_q7, 3))

<details><summary>Hint</summary>

Three independent failures:
- KMeans on raw `X` lets the column with the biggest range dominate the distance metric. Fit on the **scaled** version (`X_scaled` from Q2).
- `n_clusters` should reflect the data. Iris has 3 species; even without that knowledge, the silhouette score peaks at 2 or 3 — not 2 by accident, but by what the data says.
- When you build the crosstab, the labels variable must come from the **same fit** you care about. Mixing labels from two different KMeans calls gives garbage.

</details>

---
## Solutions

<details><summary>Show all solutions</summary>

```python
# Q1
km = KMeans(n_clusters=3, random_state=42, n_init=10)
labels = km.fit_predict(X)

# Q2
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
labels_scaled = KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(X_scaled)

# Q3 — elbow
ks = list(range(1, 9))
inertias = []
for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_scaled)
    inertias.append(km.inertia_)

# Q4 — silhouette
ks = list(range(2, 9))
sils = []
for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_scaled)
    sils.append(silhouette_score(X_scaled, km.labels_))
best_k = ks[int(np.argmax(sils))]

# Q5 — PCA scatter
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
km = KMeans(n_clusters=best_k, random_state=42, n_init=10)
labels = km.fit_predict(X_scaled)

# Q6 — sanity check
ct = pd.crosstab(y_true, labels)

# Q7 — fixes
#   - fit on X_scaled, not raw X
#   - n_clusters should be best_k (silhouette winner = 3)
#   - use the labels from the same fit
km_fixed = KMeans(n_clusters=best_k, random_state=42, n_init=10)
labels_fixed = km_fixed.fit_predict(X_scaled)
ct_fixed = pd.crosstab(y_true, labels_fixed)
purity_q7 = ct_fixed.max(axis=1).sum() / len(y_true)
```
</details>